In [1]:
import json

rows = [json.loads(line) for line in open("data/train.jsonl", encoding="utf-8")]

print("number of lines:", len(rows))
rows[0]

number of lines: 478


{'id': 2,
 'text': 'De sålde deras hus förra året.',
 'score': 0.45,
 'label': 'poor',
 'error': "wrong pronoun form ('sitt' -> 'deras')",
 'group': 434}

In [2]:
# just putting eerything in a dataframe
import pandas as pd

df = pd.DataFrame(rows)

df.head(10)

,id,text,score,label,error,group
0,2,De sålde deras hus förra året.,0.45,poor,wrong pronoun form ('sitt' -> 'deras'),434
1,4,Vi måste vänta för bussen i tjugo minuter.,0.40,poor,wrong preposition (vänta på),4
2,5,Trots att vädret var dåligt kom nästan alla ti...,0.95,good,NaN,5
3,6,Om symtomen inte går över inom en vecka bör du...,0.95,good,NaN,6
4,8,Förra året reste vi till Japan.,0.95,good,NaN,8
5,9,Företaget har anställde tjugo nya medarbetarer...,0.20,poor,wrong verb form ('anställt' -> 'anställde'); w...,141
6,10,"Efter jobbet gick jag till gymmet, men det var...",0.95,good,NaN,10
7,11,Jag bor på Sverige.,0.40,poor,wrong preposition with country (i Sverige),11
8,12,Hon köpte en ny vinter jacka.,0.45,poor,compound written as two words ('vinterjacka' -...,423
9,13,Ett barn satt på bänken och grät.,0.95,good,NaN,13


In [3]:
df.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
good,239.0,0.953138,0.021996,0.8,0.95,0.95,0.95,1.00
poor,239.0,0.350209,0.100889,0.0,0.30,0.40,0.40,0.55


In [4]:
df["score"].round(2).value_counts().sort_index()

score
0.00      1
0.05      4
0.10      4
0.15      7
0.20     20
0.25     16
0.30      9
0.35     54
0.40     73
0.45     47
0.50      2
0.55      2
0.80      1
0.85      1
0.90      7
0.95    203
1.00     27
Name: count, dtype: int64

In [5]:
# errors
poor = df[df["label"] == "poor"].copy() # only poor

for e in poor["error"].head(10):
    print(repr(e))

"wrong pronoun form ('sitt' -> 'deras')"
'wrong preposition (vänta på)'
"wrong verb form ('anställt' -> 'anställde'); wrong plural form ('medarbetare' -> 'medarbetarer')"
'wrong preposition with country (i Sverige)'
"compound written as two words ('vinterjacka' -> 'vinter jacka')"
'missing definite (modellen); compound split; agreement (själva)'
"compound written as two words ('tandborsten' -> 'tand borsten')"
'V2 violation; gender (ett gemensamt avtal); negation placement in subordinate clause'
"adverb/negation placement ('har fortfarande inte' -> 'har inte fortfarande')"
'wrong gender (ett förslag)'


In [6]:
poor["n_errors"] = poor["error"].apply(lambda e: len([p for p in e.split(";") if p.strip()]))

In [7]:
poor[["error", "n_errors"]].head(10)

,error,n_errors
0,wrong pronoun form ('sitt' -> 'deras'),1
1,wrong preposition (vänta på),1
5,wrong verb form ('anställt' -> 'anställde'); w...,2
7,wrong preposition with country (i Sverige),1
8,compound written as two words ('vinterjacka' -...,1
10,missing definite (modellen); compound split; a...,3
13,compound written as two words ('tandborsten' -...,1
15,V2 violation; gender (ett gemensamt avtal); ne...,3
16,adverb/negation placement ('har fortfarande in...,1
21,wrong gender (ett förslag),1


In [8]:
poor.groupby("n_errors")["score"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
n_errors,,,,
1,184,0.392663,0.00,0.55
2,47,0.227660,0.10,0.40
3,8,0.093750,0.05,0.15
